# Phase 1 Task B — ConvNeXt-Tiny × 3 Seeds

**Model:** ConvNeXt-Tiny
**Seeds:** 42, 123, 456
**Batch:** 32
**Output:** wandb `woodbench-taskB-clean-clean`



In [ ]:
from pathlib import Path
NOTEBOOK = Path("__file__").resolve()
ROOT = NOTEBOOK
for _ in range(10):
    ROOT = ROOT.parent
    if (ROOT / "pyproject.toml").exists() or (ROOT / ".git").exists(): break
import sys, os
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print(f"[ROOT] {ROOT}")

In [ ]:
# Load .env for API keys
from dotenv import load_dotenv
dotenv_path = ROOT / ".env"
load_dotenv(dotenv_path)
print(f"[.env] Loaded from {dotenv_path}")

import subprocess, sys
for pkg in ["timm", "albumentations", "scikit-learn", "seaborn", "wandb", "ptflops", "python-dotenv"]:
    try: __import__(pkg.replace("-", "_"))
    except: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

os.environ.setdefault("WANDB_API_KEY", os.getenv("WANDB_API_KEY", ""))
os.environ.setdefault("HF_TOKEN", os.getenv("HF_TOKEN", ""))
print(f"[wandb] API key: {'set' if os.getenv('WANDB_API_KEY') else 'MISSING'}")
print(f"[HF] token: {'set' if os.getenv('HF_TOKEN') else 'MISSING'}")

In [ ]:
# Download data from HuggingFace
HF_REPO = "ny0g507h3p/wood"
DATA_ROOT = ROOT / "data" / "task-B-group-split"

if not (DATA_ROOT / "train").exists():
    print(f"[HF] Downloading dataset to {DATA_ROOT}...")
    from huggingface_hub import login, snapshot_download
    token = os.getenv("HF_TOKEN", "")
    if token:
        login(token=token)
        print("[HF] Logged in")
    else:
        print("[HF] No token — trying public access")
    snapshot_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        local_dir=str(DATA_ROOT),
        ignore_patterns=["*.md", "README*"],
    )
    print("[HF] Download complete")
else:
    print(f"[Data] Found at {DATA_ROOT}")

# Verify
train_dir = DATA_ROOT / "train"
if not train_dir.exists():
    raise FileNotFoundError(f"[ERROR] train/ not found in {DATA_ROOT}. Dataset download failed.")
classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
print(f"[Data] {len(classes)} classes: {classes[:3]}...")
print(f"[Data] Train samples: {sum(1 for _ in train_dir.rglob('*.jpg')) + sum(1 for _ in train_dir.rglob('*.png'))}")

In [ ]:
from src.data import get_dataloaders
# num_workers=4 to avoid multiprocessing issues on vast.ai
tl, vl, testl = get_dataloaders(str(DATA_ROOT), batch_size=32, num_workers=4, task="taskB")
print(f"[Data] Train={len(tl.dataset)}, Val={len(vl.dataset)}, Test={len(testl.dataset)}")
print(f"[Data] Classes={len(tl.dataset.classes)}")

In [ ]:
import wandb, torch, numpy as np
from src.models import create_model
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function
from src.utils import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}")

In [ ]:
def train_one(backbone, seed, data_root, batch_size=32, epochs=50):
    set_seed(seed)
    output_dir = (
        ROOT / "data" / "experiments" / "Task-B" / "main" / f"seed_{seed}" / backbone
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    # Keep test_loader for inline test evaluation
    train_loader, val_loader, test_loader = get_dataloaders(
        data_root=data_root, batch_size=batch_size, num_workers=4, task="taskB"
    )
    num_classes = len(train_loader.dataset.classes)
    print(
        f"[{backbone} s{seed}] train={len(train_loader.dataset)}, classes={num_classes}"
    )

    tracker = WandbTracker.start(
        project="woodbench-taskB-clean",
        name=f"{backbone}_s{seed}",
        config={
            "backbone": backbone,
            "seed": seed,
            "epochs": epochs,
            "batch_size": batch_size,
            "num_classes": num_classes,
            "lr_head": 1e-4,
            "lr_backbone": 1e-5,
            "label_smoothing": 0.1,
            "scheduler": "cosine",
            "amp": True,
        },
        tags=[backbone, f"seed{seed}"],
        dir_path=str(output_dir),
    )

    model = create_model(backbone, num_classes=num_classes, pretrained=True)
    optimizer = create_optimizer(model, lr=1e-4, weight_decay=1e-4, backbone_lr=1e-5)
    scheduler = create_scheduler(optimizer, epochs=epochs, scheduler_type="cosine")
    criterion = create_loss_function("label_smoothing", label_smoothing=0.1)

    engine = TrainingEngine(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=DEVICE,
        use_amp=True,
    )
    result = engine.fit(
        epochs=epochs,
        output_dir=output_dir,
        name=f"{backbone}_seed{seed}",
        save_best_only=True,
        early_stopping_patience=10,
        tracker=tracker,
    )

    # ── Inline test evaluation ───────────────────────────────────────────────
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_metrics = {
        "test_accuracy": accuracy_score(all_labels, all_preds),
        "test_f1_macro": f1_score(
            all_labels, all_preds, average="macro", zero_division=0
        ),
        "test_precision_macro": precision_score(
            all_labels, all_preds, average="macro", zero_division=0
        ),
        "test_recall_macro": recall_score(
            all_labels, all_preds, average="macro", zero_division=0
        ),
    }

    # ── Log summary metrics (training + test) to wandb ──────────────────────
    if tracker:
        tracker.log(
            {
                "best_epoch": result["best_epoch"],
                "best_val_acc": result["best_val_acc"],
                "epochs_trained": result["epochs_trained"],
                "stopped_early": result["stopped_early"],
                **test_metrics,
            },
            step=result["epochs_trained"],
        )
        tracker.finish()

    # ── Persist test metrics to disk ─────────────────────────────────────────
    with open(output_dir / "test_metrics.json", "w") as fh:
        json.dump(test_metrics, fh, indent=2)

    print(f"[{backbone} s{seed}] val_acc={result['best_val_acc']:.2f}%")
    print(
        f"[{backbone} s{seed}] test_acc={test_metrics['test_accuracy']*100:.2f}%  "
        f"F1={test_metrics['test_f1_macro']*100:.2f}%"
    )
    return {**result, "test_metrics": test_metrics}

In [ ]:
# ── Cell 7: Training Loop (3 seeds) ──────────────────────────────────────────
SEEDS = [42, 123, 456]
results = {}

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"ConvNeXt-Tiny | seed={seed} ({SEEDS.index(seed)+1}/{len(SEEDS)})")
    print(f"{'='*60}")
    results[seed] = train_one("convnext_tiny", seed, str(DATA_ROOT), batch_size=32, epochs=50)

print("\n" + "="*60)
print("ConvNeXt-Tiny — Summary")
print(f"{'seed':>6}  {'val_acc':>8}  {'test_acc':>9}  {'test_f1':>8}  {'prec':>7}  {'rec':>7}")
print("-"*60)
for seed, res in results.items():
    tm = res["test_metrics"]
    print(
        f"{seed:>6}  "
        f"{res['best_val_acc']:>7.2f}%  "
        f"{tm['test_accuracy']*100:>8.2f}%  "
        f"{tm['test_f1_macro']*100:>7.2f}%  "
        f"{tm['test_precision_macro']*100:>6.2f}%  "
        f"{tm['test_recall_macro']*100:>6.2f}%"
    )
val_accs  = [r['best_val_acc'] for r in results.values()]
test_accs = [r['test_metrics']['test_accuracy']*100 for r in results.values()]
test_f1s  = [r['test_metrics']['test_f1_macro']*100 for r in results.values()]
print("-"*60)
print(f"{'mean':>6}  {np.mean(val_accs):>7.2f}%  {np.mean(test_accs):>8.2f}%  {np.mean(test_f1s):>7.2f}%")
print(f"{'std':>6}  {np.std(val_accs):>7.2f}%  {np.std(test_accs):>8.2f}%  {np.std(test_f1s):>7.2f}%")


In [ ]:
# ── Cell 8: Confusion Matrix + Test Metrics (all seeds) ──────────────────────
import torch, numpy as np, matplotlib.pyplot as plt, wandb
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from src.data import get_dataloaders

DATA_ROOT = ROOT / "data" / "task-B-group-split"
_ignored_train, _ignored_val, test_loader = get_dataloaders(
    str(DATA_ROOT), batch_size=32, num_workers=4, task="taskB"
)
class_names = test_loader.dataset.classes
num_classes = len(class_names)
print(f"[CM] {num_classes} classes: {class_names}")

all_preds, all_labels = {}, {}

for seed in SEEDS:
    backbone_dir = ROOT / "data" / "experiments" / "Task-B" / "main" / f"seed_{seed}" / "convnext_tiny"

    ckpt_candidates = sorted(backbone_dir.glob("*best*.pth"))
    if not ckpt_candidates:
        print(f"[CM] No checkpoint found for seed={seed}, skipping")
        continue
    ckpt_path = ckpt_candidates[0]
    print(f"[CM] Loading: {ckpt_path}")

    from src.models import create_model

    model = create_model("convnext_tiny", num_classes=num_classes, pretrained=False)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(DEVICE).eval()

    preds_s, labels_s = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds_s.extend(outputs.argmax(1).cpu().numpy())
            labels_s.extend(labels.numpy())
    all_preds[seed] = np.array(preds_s)
    all_labels[seed] = np.array(labels_s)

    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

    acc = accuracy_score(labels_s, preds_s)
    f1 = f1_score(labels_s, preds_s, average="macro", zero_division=0)
    prec = precision_score(labels_s, preds_s, average="macro", zero_division=0)
    rec = recall_score(labels_s, preds_s, average="macro", zero_division=0)
    print(
        f"[CM s{seed}] Acc={acc*100:.2f}%  F1={f1*100:.2f}%  Prec={prec*100:.2f}%  Rec={rec*100:.2f}%"
    )

    f1_per = f1_score(labels_s, preds_s, average=None, zero_division=0)
    for cn, f in zip(class_names, f1_per):
        print(f"  {cn}: F1={f*100:.2f}%")

    wandb.init(
        project="woodbench-taskB-clean",
        name=f"convnext_tiny_s{seed}_test",
        dir=str(backbone_dir),
    )
    wandb.log(
        {
            "test/accuracy": acc,
            "test/precision": prec,
            "test/recall": rec,
            "test/f1": f1,
            **{f"test/f1/{cn}": float(fi) for cn, fi in zip(class_names, f1_per)},
        }
    )
    wandb.finish()

# ── Confusion Matrix Plot ────────────────────────────────────────────────────
if not all_preds:
    raise RuntimeError("No checkpoints found — cannot plot confusion matrix")

best_seed = max(all_preds, key=lambda s: accuracy_score(all_labels[s], all_preds[s]))
y_true, y_pred = all_labels[best_seed], all_preds[best_seed]

fig, ax = plt.subplots(figsize=(18, 14))
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    display_labels=class_names,
    cmap="Blues",
    normalize="true",
    ax=ax,
    values_format=".2f",
)
ax.set_title(
    f"ConvNeXt-Tiny — Confusion Matrix (Test Set, seed={best_seed})", fontsize=14
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

cm_dir = ROOT / "data" / "experiments" / "Task-B"/ "main" / f"seed_{seed}" / "convnext_tiny"
cm_path = cm_dir / "confusion_matrix_convnext_tiny.png"
fig.savefig(cm_path, dpi=150, bbox_inches="tight")
print(f"[CM] Saved to {cm_path}")

wandb.init(project="woodbench-taskB-clean", name="convnext_tiny_CM", dir=str(cm_dir))
wandb.log({"confusion_matrix": wandb.Image(str(cm_path))})
wandb.finish()
plt.show()

print("\n" + "=" * 80)
print("Classification Report (best seed =", best_seed, ")")
print("=" * 80)
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
import os, sys
print("PID:", os.getpid())
print("Python:", sys.executable)